<a href="https://colab.research.google.com/github/Kaizen-es/healthcare-machine-learning-pipeline/blob/main/healthpipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np
import plotly.express as px
import kagglehub
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix


from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler


In [2]:
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")


Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.


In [3]:
# import data
df = pd.read_csv(path + "/data.csv")
pd.set_option('display.max_columns', None)
# data exploration
df.head()
df.columns
df.shape


(569, 33)

In [4]:
print(df['Unnamed: 32'])
#print(df.describe())
df.info()


0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
       ..
564   NaN
565   NaN
566   NaN
567   NaN
568   NaN
Name: Unnamed: 32, Length: 569, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 no

In [5]:
df.duplicated().sum()
df.isnull().sum()

,0
id,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0


In [6]:
# Data Cleaning
df_clean = pd.read_csv(path + "/data.csv")
df_clean.drop(['Unnamed: 32','id'], axis=1, inplace=True)
df_clean.columns
df_clean.shape

(569, 31)

In [7]:
BENIGN_COLOR = '#2ecc71'
MALIGNANT_COLOR = '#e74c3c'

benign = df_clean[df_clean['diagnosis'] == 'B']
malignant = df_clean[df_clean['diagnosis'] == 'M']

mean_features = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
    'smoothness_mean', 'compactness_mean', 'concavity_mean',
    'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean'
]

# ── 1. radius_mean vs radius_se vs radius_worst (histograms) ──────────────
radius_variants = ['radius_mean', 'radius_se', 'radius_worst']

fig0 = make_subplots(rows=1, cols=3, subplot_titles=radius_variants)

for i, feat in enumerate(radius_variants):
    fig0.add_trace(go.Histogram(
        x=benign[feat], name='Benign', legendgroup='benign',
        marker_color=BENIGN_COLOR, opacity=0.6,
        showlegend=(i == 0)
    ), row=1, col=i+1)
    fig0.add_trace(go.Histogram(
        x=malignant[feat], name='Malignant', legendgroup='malignant',
        marker_color=MALIGNANT_COLOR, opacity=0.6,
        showlegend=(i == 0)
    ), row=1, col=i+1)

fig0.update_layout(
    title='Radius: mean vs se vs worst',
    barmode='overlay',
    template='plotly_white',
    height=350
)
fig0.show()

# ── helper to build KDE trace ─────────────────────────────────────────────
def kde_trace(data, color, name, showlegend=True):
    x = np.linspace(data.min(), data.max(), 200)
    y = gaussian_kde(data)(x)
    return go.Scatter(
        x=x, y=y, mode='lines', name=name,
        fill='tozeroy', line=dict(color=color, width=2),
        opacity=0.5, showlegend=showlegend
    )

# ── 2. Histograms — 10 mean features ─────────────────────────────────────
fig1 = make_subplots(rows=2, cols=5, subplot_titles=mean_features)

for i, feat in enumerate(mean_features):
    r, c = divmod(i, 5)
    show = (i == 0)
    fig1.add_trace(go.Histogram(
        x=benign[feat], name='Benign', legendgroup='benign',
        marker_color=BENIGN_COLOR, opacity=0.6, showlegend=show
    ), row=r+1, col=c+1)
    fig1.add_trace(go.Histogram(
        x=malignant[feat], name='Malignant', legendgroup='malignant',
        marker_color=MALIGNANT_COLOR, opacity=0.6, showlegend=show
    ), row=r+1, col=c+1)

fig1.update_layout(
    title='Histograms — mean features',
    barmode='overlay',
    template='plotly_white',
    height=500
)
fig1.show()

# ── 3. KDE — 10 mean features ─────────────────────────────────────────────
fig2 = make_subplots(rows=2, cols=5, subplot_titles=mean_features)

for i, feat in enumerate(mean_features):
    r, c = divmod(i, 5)
    show = (i == 0)
    fig2.add_trace(kde_trace(benign[feat], BENIGN_COLOR, 'Benign', show), row=r+1, col=c+1)
    fig2.add_trace(kde_trace(malignant[feat], MALIGNANT_COLOR, 'Malignant', show), row=r+1, col=c+1)

fig2.update_layout(
    title='KDE Curves — mean features',
    template='plotly_white',
    height=500
)
fig2.show()

# ── 4. Box plots — 10 mean features ──────────────────────────────────────
fig3 = make_subplots(rows=2, cols=5, subplot_titles=mean_features)

for i, feat in enumerate(mean_features):
    r, c = divmod(i, 5)
    show = (i == 0)
    fig3.add_trace(go.Box(
        y=benign[feat], name='Benign', legendgroup='benign',
        marker_color=BENIGN_COLOR, showlegend=show
    ), row=r+1, col=c+1)
    fig3.add_trace(go.Box(
        y=malignant[feat], name='Malignant', legendgroup='malignant',
        marker_color=MALIGNANT_COLOR, showlegend=show
    ), row=r+1, col=c+1)

fig3.update_layout(
    title='Box Plots — mean features',
    template='plotly_white',
    height=500
)
fig3.show()

In [8]:
# Data visualization
px.pie(df_clean, names='diagnosis', title='Class Distribution')

In [9]:
# Create a copy for classification and encode target variable
df_class = df_clean.copy()
df_class['diagnosis'] = df_class['diagnosis'].map({'M': 1, 'B': 0})


In [10]:
# Train Test Split
X = df_class.drop('diagnosis', axis=1)
y = df_class['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=5642, stratify =y)

In [11]:
# Scaling X
Scaler = StandardScaler()
X_train_scaled = Scaler.fit_transform(X_train)
X_test_scaled = Scaler.transform(X_test)

In [12]:
LR = LogisticRegression(random_state=5642)
SVM = SVC(kernel='linear', C=1, random_state=5642, probability=True)
RF = RandomForestClassifier(random_state=5642)

LR.fit(X_train_scaled, y_train)
SVM.fit(X_train_scaled, y_train)
RF.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=5642)

In [13]:
y_p_LR = LR.predict(X_test_scaled)
y_p_SVM = SVM.predict(X_test_scaled)
y_p_RF = RF.predict(X_test_scaled)

In [14]:
print('Logistic Regression:', accuracy_score(y_test, y_p_LR))
print('SVM:', accuracy_score(y_test, y_p_SVM))
print('Random Forest:', accuracy_score(y_test, y_p_RF))

Logistic Regression: 0.9824561403508771
SVM: 0.9736842105263158
Random Forest: 0.956140350877193


In [15]:
print('Logistic Regression')
print(classification_report(y_test, y_p_LR))
print('SVM')
print(classification_report(y_test, y_p_SVM))
print('Random Forest')
print(classification_report(y_test, y_p_RF))

Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        72
           1       0.95      1.00      0.98        42

    accuracy                           0.98       114
   macro avg       0.98      0.99      0.98       114
weighted avg       0.98      0.98      0.98       114

SVM
              precision    recall  f1-score   support

           0       0.99      0.97      0.98        72
           1       0.95      0.98      0.96        42

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

Random Forest
              precision    recall  f1-score   support

           0       0.96      0.97      0.97        72
           1       0.95      0.93      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96   

In [16]:
# Confusion Matrix
classifiers = {'Logistic Regression': y_p_LR,
               'SVM': y_p_SVM,
               'Random Forest': y_p_RF}

fig4 = make_subplots(rows=1, cols=3,
                     subplot_titles=list(classifiers.keys()))

for i, (name, y_pred) in enumerate(classifiers.items()):
    cm = confusion_matrix(y_test, y_pred)
    cm_colors = [[1, 0],   # TN=correct, FP=error
                [0, 1]]   # FN=error, TP=correct

    fig4.add_trace(go.Heatmap(
        z=cm_colors,
        x=['Benign', 'Malignant'],
        y=['Benign', 'Malignant'],
        colorscale=[[0, '#e74c3c'], [1, '#2ecc71']],
        text=cm,
        texttemplate='%{text}',
        showscale=False,
        hoverinfo='none'
    ), row=1, col=i+1)

fig4.update_layout(
    title='Confusion Matrices — Baseline',
    template='plotly_white',
    height=400
)
fig4.show()

In [17]:
print(X_train.shape)
print(y_train.value_counts())

(455, 30)
diagnosis
0    285
1    170
Name: count, dtype: int64


In [18]:
# Case 1: 20% malignant, 80% benign (91 malignant, 364 benign)
over = RandomOverSampler(sampling_strategy={0: 364}, random_state=5642)
X_case1, y_case1 = over.fit_resample(X_train, y_train)

under = RandomUnderSampler(sampling_strategy={1: 91}, random_state=5642)
X_case1, y_case1 = under.fit_resample(X_case1, y_case1)

print(y_case1.value_counts())

diagnosis
0    364
1     91
Name: count, dtype: int64


In [19]:
# Case 2: 50% malignant, 50% benign (228 malignant, 227 benign)
over = RandomOverSampler(sampling_strategy={1: 228}, random_state=5642)
X_case2, y_case2 = over.fit_resample(X_train, y_train)

under = RandomUnderSampler(sampling_strategy={0: 227}, random_state=5642)
X_case2, y_case2 = under.fit_resample(X_case2, y_case2)

print(y_case2.value_counts())

diagnosis
1    228
0    227
Name: count, dtype: int64


In [20]:
# Case 3: 80% malignant, 20% benign (91 malignant, 364 benign)
over = RandomOverSampler(sampling_strategy={1: 364}, random_state=5642)
X_case3, y_case3 = over.fit_resample(X_train, y_train)

under = RandomUnderSampler(sampling_strategy={0: 91}, random_state=5642)
X_case3, y_case3 = under.fit_resample(X_case3, y_case3)

print(y_case3.value_counts())

diagnosis
1    364
0     91
Name: count, dtype: int64


In [21]:
# Scaling X the other classes
X_case1_scaled = Scaler.transform(X_case1)
X_case2_scaled = Scaler.transform(X_case2)
X_case3_scaled = Scaler.transform(X_case3)

In [22]:
LR.fit(X_case1_scaled, y_case1)
SVM.fit(X_case1_scaled, y_case1)
RF.fit(X_case1_scaled, y_case1)

y_p_LR_case1 = LR.predict(X_test_scaled)
y_p_SVM_case1 = SVM.predict(X_test_scaled)
y_p_RF_case1 = RF.predict(X_test_scaled)

In [23]:
print('Logistic Regression:', accuracy_score(y_test, y_p_LR_case1))
print('SVM:', accuracy_score(y_test, y_p_SVM_case1))
print('Random Forest:', accuracy_score(y_test, y_p_RF_case1))

Logistic Regression: 0.9649122807017544
SVM: 0.956140350877193
Random Forest: 0.9473684210526315


In [24]:
LR.fit(X_case2_scaled, y_case2)
SVM.fit(X_case2_scaled, y_case2)
RF.fit(X_case2_scaled, y_case2)

y_p_LR_case2 = LR.predict(X_test_scaled)
y_p_SVM_case2 = SVM.predict(X_test_scaled)
y_p_RF_case2 = RF.predict(X_test_scaled)

In [25]:
print('Logistic Regression:', accuracy_score(y_test, y_p_LR_case2))
print('SVM:', accuracy_score(y_test, y_p_SVM_case2))
print('Random Forest:', accuracy_score(y_test, y_p_RF_case2))

Logistic Regression: 0.9736842105263158
SVM: 0.9649122807017544
Random Forest: 0.956140350877193


In [26]:
LR.fit(X_case3_scaled, y_case3)
SVM.fit(X_case3_scaled, y_case3)
RF.fit(X_case3_scaled, y_case3)

y_p_LR_case3 = LR.predict(X_test_scaled)
y_p_SVM_case3 = SVM.predict(X_test_scaled)
y_p_RF_case3 = RF.predict(X_test_scaled)

In [27]:
print('Logistic Regression:', accuracy_score(y_test, y_p_LR_case3))
print('SVM:', accuracy_score(y_test, y_p_SVM_case3))
print('Random Forest:', accuracy_score(y_test, y_p_RF_case3))

Logistic Regression: 0.9122807017543859
SVM: 0.9035087719298246
Random Forest: 0.9210526315789473


In [28]:
cases = ['Baseline (37/63)', 'Case 1 (20/80)', 'Case 2 (50/50)', 'Case 3 (80/20)']

LR_scores = [0.982, 0.965, 0.974, 0.912]
SVM_scores = [0.974, 0.956, 0.965, 0.904]
RF_scores = [0.956, 0.947, 0.956, 0.921]

fig5 = go.Figure()

fig5.add_trace(go.Scatter(
    x=cases, y=LR_scores, mode='lines+markers',
    name='Logistic Regression', line=dict(color='#3498db', width=2)
))
fig5.add_trace(go.Scatter(
    x=cases, y=SVM_scores, mode='lines+markers',
    name='SVM', line=dict(color='#e74c3c', width=2)
))
fig5.add_trace(go.Scatter(
    x=cases, y=RF_scores, mode='lines+markers',
    name='Random Forest', line=dict(color='#2ecc71', width=2)
))

fig5.update_layout(
    title='Accuracy Across Class Distribution Cases',
    xaxis_title='Class Distribution',
    yaxis_title='Accuracy',
    template='plotly_white',
    yaxis=dict(range=[0.85, 1.0]),
    height=400
)
fig5.show()

In [29]:
from sklearn.metrics import roc_auc_score

cases = {
    'Baseline (37/63)': (y_p_LR, y_p_SVM, y_p_RF),
    'Case 1 (20/80)': (y_p_LR_case1, y_p_SVM_case1, y_p_RF_case1),
    'Case 2 (50/50)': (y_p_LR_case2, y_p_SVM_case2, y_p_RF_case2),
    'Case 3 (80/20)': (y_p_LR_case3, y_p_SVM_case3, y_p_RF_case3)
}

classifiers = ['Logistic Regression', 'SVM', 'Random Forest']
case_names = list(cases.keys())

results = {clf: {'sensitivity': [], 'specificity': [], 'auc': []} for clf in classifiers}

# sensitivity and specificity
for case_name, preds in cases.items():
    for clf_name, y_pred in zip(classifiers, preds):
        cm = confusion_matrix(y_test, y_pred)
        TN, FP, FN, TP = cm.ravel()
        results[clf_name]['sensitivity'].append(TP / (TP + FN))
        results[clf_name]['specificity'].append(TN / (TN + FP))

# AUC-ROC
LR = LogisticRegression(random_state=5642, max_iter=10000)
SVM = SVC(kernel='linear', C=1, random_state=5642, probability=True)
RF = RandomForestClassifier(random_state=5642)

model_case_map = [
    (X_train_scaled, y_train),
    (X_case1_scaled, y_case1),
    (X_case2_scaled, y_case2),
    (X_case3_scaled, y_case3)
]

for i, (X_tr, y_tr) in enumerate(model_case_map):
    for clf, clf_name in zip([LR, SVM, RF], classifiers):
        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        results[clf_name]['auc'].append(auc)

# print results
for clf_name in classifiers:
    print(f'\n{clf_name}')
    for i, case in enumerate(case_names):
        print(f"  {case} — Sensitivity: {results[clf_name]['sensitivity'][i]:.3f}, "
              f"Specificity: {results[clf_name]['specificity'][i]:.3f}, "
              f"AUC: {results[clf_name]['auc'][i]:.3f}")


Logistic Regression
  Baseline (37/63) — Sensitivity: 1.000, Specificity: 0.972, AUC: 0.998
  Case 1 (20/80) — Sensitivity: 0.929, Specificity: 0.986, AUC: 0.996
  Case 2 (50/50) — Sensitivity: 1.000, Specificity: 0.958, AUC: 0.998
  Case 3 (80/20) — Sensitivity: 1.000, Specificity: 0.861, AUC: 0.998

SVM
  Baseline (37/63) — Sensitivity: 0.976, Specificity: 0.972, AUC: 0.997
  Case 1 (20/80) — Sensitivity: 0.905, Specificity: 0.986, AUC: 0.995
  Case 2 (50/50) — Sensitivity: 0.976, Specificity: 0.958, AUC: 0.997
  Case 3 (80/20) — Sensitivity: 1.000, Specificity: 0.847, AUC: 0.993

Random Forest
  Baseline (37/63) — Sensitivity: 0.929, Specificity: 0.972, AUC: 0.993
  Case 1 (20/80) — Sensitivity: 0.857, Specificity: 1.000, AUC: 0.975
  Case 2 (50/50) — Sensitivity: 0.952, Specificity: 0.958, AUC: 0.993
  Case 3 (80/20) — Sensitivity: 1.000, Specificity: 0.875, AUC: 0.992


In [30]:
# ── Metric Line Charts ────────────────────────────────────────────────────
cases = ['Baseline (37/63)', 'Case 1 (20/80)', 'Case 2 (50/50)', 'Case 3 (80/20)']
colors = {'Logistic Regression': '#3498db', 'SVM': '#e74c3c', 'Random Forest': '#2ecc71'}
metrics = ['sensitivity', 'specificity', 'auc']
titles = ['Sensitivity Across Class Distribution Cases',
          'Specificity Across Class Distribution Cases',
          'AUC-ROC Across Class Distribution Cases']
yranges = [[0.80, 1.01], [0.80, 1.01], [0.95, 1.01]]

for metric, title, yrange in zip(metrics, titles, yranges):
    fig = go.Figure()
    for clf_name in ['Logistic Regression', 'SVM', 'Random Forest']:
        fig.add_trace(go.Scatter(
            x=cases,
            y=results[clf_name][metric],
            mode='lines+markers',
            name=clf_name,
            line=dict(color=colors[clf_name], width=2)
        ))
    fig.update_layout(
        title=title,
        xaxis_title='Class Distribution',
        yaxis_title=metric.upper(),
        template='plotly_white',
        yaxis=dict(range=yrange),
        height=400
    )
    fig.show()

# ── Confusion Matrices for All Cases ─────────────────────────────────────
cm_colors = [[1, 0], [0, 1]]
colorscale = [[0, '#e74c3c'], [1, '#2ecc71']]

case_preds = {
    'Case 1 (20/80)': (y_p_LR_case1, y_p_SVM_case1, y_p_RF_case1),
    'Case 2 (50/50)': (y_p_LR_case2, y_p_SVM_case2, y_p_RF_case2),
    'Case 3 (80/20)': (y_p_LR_case3, y_p_SVM_case3, y_p_RF_case3)
}

for case_name, preds in case_preds.items():
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=['Logistic Regression', 'SVM', 'Random Forest'])
    for i, y_pred in enumerate(preds):
        cm = confusion_matrix(y_test, y_pred)
        fig.add_trace(go.Heatmap(
            z=cm_colors,
            x=['Benign', 'Malignant'],
            y=['Benign', 'Malignant'],
            colorscale=colorscale,
            text=cm,
            texttemplate='%{text}',
            showscale=False,
            hoverinfo='none'
        ), row=1, col=i+1)
    fig.update_layout(
        title=f'Confusion Matrices — {case_name}',
        template='plotly_white',
        height=400
    )
    fig.show()

In [31]:
from sklearn.metrics import confusion_matrix
import numpy as np

cases = ['Baseline (37/63)', 'Case 1 (20/80)', 'Case 2 (50/50)', 'Case 3 (80/20)']
classifiers = ['Logistic Regression', 'SVM', 'Random Forest']

all_preds = {
    'Baseline (37/63)': (y_p_LR, y_p_SVM, y_p_RF),
    'Case 1 (20/80)':   (y_p_LR_case1, y_p_SVM_case1, y_p_RF_case1),
    'Case 2 (50/50)':   (y_p_LR_case2, y_p_SVM_case2, y_p_RF_case2),
    'Case 3 (80/20)':   (y_p_LR_case3, y_p_SVM_case3, y_p_RF_case3)
}

print("STREAMLIT HARDCODED VALUES")

print("\naccuracy = {")
for case, preds in all_preds.items():
    vals = [round(accuracy_score(y_test, p), 4) for p in preds]
    print(f"    '{case}': {{'Logistic Regression': {vals[0]}, 'SVM': {vals[1]}, 'Random Forest': {vals[2]}}},")
print("}")

print("\nsensitivity = {")
for case, preds in all_preds.items():
    vals = []
    for p in preds:
        TN, FP, FN, TP = confusion_matrix(y_test, p).ravel()
        vals.append(round(TP / (TP + FN), 4))
    print(f"    '{case}': {{'Logistic Regression': {vals[0]}, 'SVM': {vals[1]}, 'Random Forest': {vals[2]}}},")
print("}")

print("\nspecificity = {")
for case, preds in all_preds.items():
    vals = []
    for p in preds:
        TN, FP, FN, TP = confusion_matrix(y_test, p).ravel()
        vals.append(round(TN / (TN + FP), 4))
    print(f"    '{case}': {{'Logistic Regression': {vals[0]}, 'SVM': {vals[1]}, 'Random Forest': {vals[2]}}},")
print("}")

print("\nauc = {")
for case, preds in all_preds.items():
    vals = [round(results[clf]['auc'][i], 4)
            for i, clf in enumerate(classifiers)]
    # fix indexing
    case_idx = list(all_preds.keys()).index(case)
    vals = [round(results[clf]['auc'][case_idx], 4) for clf in classifiers]
    print(f"    '{case}': {{'Logistic Regression': {vals[0]}, 'SVM': {vals[1]}, 'Random Forest': {vals[2]}}},")
print("}")

print("\ncm_data = {")
for case, preds in all_preds.items():
    print(f"    '{case}': {{")
    for clf_name, p in zip(classifiers, preds):
        TN, FP, FN, TP = confusion_matrix(y_test, p).ravel()
        print(f"        '{clf_name}': {{'TN': {TN}, 'FP': {FP}, 'FN': {FN}, 'TP': {TP}}},")
    print("    },")
print("}")

print("\n# Feature stats for Stage 1")
mean_features = [
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean',
    'smoothness_mean', 'compactness_mean', 'concavity_mean',
    'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean'
]
benign = df_clean[df_clean['diagnosis'] == 'B']
malignant = df_clean[df_clean['diagnosis'] == 'M']

print("feature_data = {")
for feat in mean_features:
    b_vals = benign[feat].tolist()
    m_vals = malignant[feat].tolist()
    print(f"    '{feat}': {{")
    print(f"        'benign': {b_vals},")
    print(f"        'malignant': {m_vals}")
    print("    },")
print("}")

print("\n# Class distribution")
print(f"class_counts = {{'Benign': {len(benign)}, 'Malignant': {len(malignant)}}}")

print("\n# Training set sizes per case")
print(f"training_sizes = {{")
print(f"    'Baseline (37/63)': {{'Benign': 285, 'Malignant': 170}},")
print(f"    'Case 1 (20/80)':   {{'Benign': 364, 'Malignant': 91}},")
print(f"    'Case 2 (50/50)':   {{'Benign': 227, 'Malignant': 228}},")
print(f"    'Case 3 (80/20)':   {{'Benign': 91, 'Malignant': 364}}")
print(f"}}")

print("\n# Real world impact")
print("us_cases = 382640")
print("global_cases = 2300000")


STREAMLIT HARDCODED VALUES

accuracy = {
    'Baseline (37/63)': {'Logistic Regression': 0.9825, 'SVM': 0.9737, 'Random Forest': 0.9561},
    'Case 1 (20/80)': {'Logistic Regression': 0.9649, 'SVM': 0.9561, 'Random Forest': 0.9474},
    'Case 2 (50/50)': {'Logistic Regression': 0.9737, 'SVM': 0.9649, 'Random Forest': 0.9561},
    'Case 3 (80/20)': {'Logistic Regression': 0.9123, 'SVM': 0.9035, 'Random Forest': 0.9211},
}

sensitivity = {
    'Baseline (37/63)': {'Logistic Regression': 1.0, 'SVM': 0.9762, 'Random Forest': 0.9286},
    'Case 1 (20/80)': {'Logistic Regression': 0.9286, 'SVM': 0.9048, 'Random Forest': 0.8571},
    'Case 2 (50/50)': {'Logistic Regression': 1.0, 'SVM': 0.9762, 'Random Forest': 0.9524},
    'Case 3 (80/20)': {'Logistic Regression': 1.0, 'SVM': 1.0, 'Random Forest': 1.0},
}

specificity = {
    'Baseline (37/63)': {'Logistic Regression': 0.9722, 'SVM': 0.9722, 'Random Forest': 0.9722},
    'Case 1 (20/80)': {'Logistic Regression': 0.9861, 'SVM': 0.9861, 'Rando